# 🎨 Anima LoRA Studio (Colab)

**Anima base v1.0** LoRA 训练工作台，双选项卡 WebUI：

| 选项卡 | 做什么 | 引擎 |
|---|---|---|
| ① 打标 / Tagger | 上传图片 → WD14 **v3** 自动打标 → 标签清洗 | Dataset_Maker 的配方（kohya `tag_images_by_wd14_tagger.py` + `SmilingWolf/wd-eva02-large-tagger-v3` + `wd-vit-large-tagger-v3`） |
| ② 训练 / Trainer | 设参数 → 训 LoRA → 实时日志/产物 | [Anima-Standalone-Trainer](https://github.com/gazingstars123/Anima-Standalone-Trainer) 的 `anima_train_network.py` |

所有代码从 GitHub 拉取，本 notebook 不含任何 base64 载荷。
运行时建议选 **GPU（L4 / A100）**。

> 首次跑第 2 格会装 torch 2.7.0+cu128 等依赖，约 5–8 分钟；之后再跑只重启服务。


In [ ]:
#@title 安装/更新并启动 Anima LoRA Studio { display-mode: "form" }
import os, subprocess, pathlib

ROOT = "/content/anima-lora-studio"
CODE = f"{ROOT}/code"
REPO = "https://github.com/hsgwktb/anima-lora-studio.git"
FORCE_REINSTALL = False   #@param {type:"boolean"}

os.makedirs(ROOT, exist_ok=True)

if os.path.isdir(f"{CODE}/.git"):
    print(subprocess.run(["git", "-C", CODE, "pull", "--ff-only"],
                         capture_output=True, text=True).stdout)
else:
    print(subprocess.run(["git", "clone", "--depth", "1", REPO, CODE],
                         capture_output=True, text=True).stderr)

args = ["bash", f"{CODE}/colab_setup.sh"] + (["--install"] if FORCE_REINSTALL else [])
proc = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print("
exit", proc.returncode)


In [ ]:
#@title 打开 WebUI 地址（含 cloudflared 公网隧道）
import re, os
ROOT = "/content/anima-lora-studio"

try:
    from google.colab import output
    local = output.eval_js("google.colab.kernel.proxyPort(8000)")
    print("Colab 代理地址（推荐，稳定）:", local)
except Exception as e:
    print("(no colab proxy:", e, ")")

try:
    log = open(f"{ROOT}/tunnel.log", encoding="utf-8", errors="replace").read()
    m = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", log)
    print("公网隧道地址:", m[0] if m else "(未就绪，重跑上一格)")
except FileNotFoundError:
    print("先跑上一格")
